# S4 — Guided SPN feature selection, modelling, calibration, and robustness

Run each section in order. The notebook proposes defaults, but every decision cell can be edited before the following stage is run.


In [ ]:
from pathlib import Path
import os, sys, joblib, pandas as pd
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
EVENTS_DIR, F360_DIR, OUTPUT_DIR = Path(os.environ['EVENTS_DIR']), Path(os.environ['F360_DIR']), Path(os.environ['OUTPUT_DIR'])
table = pd.read_parquet(OUTPUT_DIR / 'spn_features.parquet')


## 1. Data and feature-quality checks


In [ ]:
from src.spn_features import SPN_ALL_FEATURES, SPN_SELECTED_FEATURES
from src.spn_model import feature_diagnostics

print('Rows:', len(table), 'matches:', table['match_id'].nunique(), 'sequences:', table[['match_id','seq_id']].drop_duplicates().shape[0])
display(table['outcome_tag'].value_counts(dropna=False).to_frame('anchors'))
diagnostics = feature_diagnostics(table, SPN_ALL_FEATURES)
display(diagnostics['missing'])
display(diagnostics['correlation_pairs'].head(20))


## 2. User feature decision

Edit exclusions or replace the proposed final SPN-top-15 list, then run the next cell.


In [ ]:
USER_EXCLUDED_FEATURES = []
FINAL_FEATURES = [feature for feature in SPN_SELECTED_FEATURES if feature not in USER_EXCLUDED_FEATURES]
print('Selected features:', FINAL_FEATURES)


## 3. Grouped model comparison

The split unit is match_id, preventing frames from the same match appearing in both training and validation folds.


In [ ]:
from src.spn_model import DEFAULT_RANDOM_STATE, FINAL_SPN_XGB_PARAMS, grouped_model_comparison

MODEL_FOLDS = 5
RANDOM_STATE = DEFAULT_RANDOM_STATE
# These are the final-main SPN-top-15 M1 defaults. Edit this dictionary for a user-data sensitivity analysis.
XGB_PARAMS = FINAL_SPN_XGB_PARAMS.copy()
comparison, fitted = grouped_model_comparison(
    table, FINAL_FEATURES, folds=MODEL_FOLDS, random_state=RANDOM_STATE, xgb_params=XGB_PARAMS
)
display(comparison)
RECOMMENDED_MODEL = comparison.iloc[0]['model']
print('Recommended by grouped log-loss:', RECOMMENDED_MODEL)


## 4. User model and calibration decision

Raw is the portable default. The table below compares raw and isotonic probabilities with nested match-grouped OOF validation: each evaluated match is excluded from both the base-model fit and the isotonic fit. A calibrator fitted on one competition or season should not be assumed to transfer to another.


In [ ]:
from src.spn_model import grouped_calibration_comparison

calibration_comparison = grouped_calibration_comparison(
    table, FINAL_FEATURES, RECOMMENDED_MODEL, folds=MODEL_FOLDS, inner_folds=3,
    random_state=RANDOM_STATE, xgb_params=XGB_PARAMS,
)
display(calibration_comparison)
RECOMMENDED_CALIBRATION = calibration_comparison.iloc[0]['calibration']
print('Recommended by nested grouped log-loss:', RECOMMENDED_CALIBRATION)


In [ ]:
SELECTED_MODEL = RECOMMENDED_MODEL  # logistic, xgb_unweighted, or xgb_balanced
CALIBRATION = 'raw'                   # raw is portable default; set isotonic only after reviewing the table above
print(SELECTED_MODEL, CALIBRATION)


In [ ]:
from src.spn_model import CLASS_NAMES, fit_isotonic, save_bundle

selected = fitted[SELECTED_MODEL]
calibrators, calibration_metadata = None, {'fit_scope': 'raw'}
if CALIBRATION == 'isotonic':
    model_rows = table['outcome_tag'].isin(CLASS_NAMES)
    y = table.loc[model_rows, 'outcome_tag'].map({name: i for i, name in enumerate(CLASS_NAMES)}).to_numpy()
    calibrators = fit_isotonic(selected['oof_probability'], y)
    calibration_metadata = {
        'fit_scope': 'match-grouped OOF raw predictions from this user dataset',
        'selection_rule': 'enable only after nested grouped validation improves the chosen probability score',
    }
bundle_path = OUTPUT_DIR / 'spn_model_bundle.joblib'
save_bundle(bundle_path, selected['model'], FINAL_FEATURES, CALIBRATION, calibrators, calibration_metadata)
print('Saved:', bundle_path)


## 5. Robustness testing

Set RUN_ROBUSTNESS to True after the model definition is fixed. Each scenario rebuilds the user-data SPN table under changed network parameters and reports feature stability.


In [ ]:
RUN_ROBUSTNESS = False
ROBUSTNESS_SCENARIOS = {
    'player_reach_low': {'player_distance': 5.76, 'player_sd': 1.98},
    'player_reach_high': {'player_distance': 7.04, 'player_sd': 2.42},
    'boundary_low': {'boundary_distance': 2.56, 'boundary_sd': .88},
    'boundary_high': {'boundary_distance': 3.84, 'boundary_sd': 1.32},
}
if RUN_ROBUSTNESS:
    from src.spn_network import PressureParams
    from src.spn_features import build_spn_feature_table
    from src.spn_model import run_robustness
    labels = pd.read_csv(OUTPUT_DIR / 'labels.csv')
    def rebuild(overrides):
        base = PressureParams()
        return build_spn_feature_table(labels, EVENTS_DIR, F360_DIR, PressureParams(**{**base.__dict__, **overrides}))
    robustness = run_robustness(table, rebuild, ROBUSTNESS_SCENARIOS, FINAL_FEATURES)
    display(robustness.groupby('scenario')[['spearman','mean_abs_change']].mean())
